# 05 Create SFINCS Scenarios

## Stage Contract

Requires: Inland event catalog, Wflow dynamic handoff readiness, and SFINCS scenario root.

Produces: Joint Wflow-SFINCS worklist and cluster launch plan.

Next: Submit the coupled Wflow-SFINCS cluster pipeline or run 04/c_run_example.ipynb for a single event.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import xarray as xr

location_root = Path("..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

from wflow_runs.notebook import load_runtime

runtime = load_runtime(location_root, wflow_domain_review_required=False)
location_name = runtime.location_name
resolve_location_path = runtime.resolve_location_path


## Package Worklist


In [ ]:
scenario_root = runtime.sfincs_scenarios_root
scenario_root.mkdir(parents=True, exist_ok=True)

catalog_path = resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv")
catalog = pd.read_csv(catalog_path, dtype={"event_id": str})

rainfall_members_path = resolve_location_path("data/sources/aorc_sst/rainfall_members.csv")
if rainfall_members_path.exists() and catalog_path.stat().st_mtime < rainfall_members_path.stat().st_mtime:
    raise RuntimeError(
        f"Scenario catalog is older than current AORC SST rainfall members: {catalog_path}. "
        "Rerun 02_flood/03_build_event_catalog.ipynb after the AORC SST Event Windows repair."
    )

historical = catalog.get("event_origin", pd.Series(index=catalog.index, dtype=object)).astype(str).eq("historical_tail")
historical |= catalog.get("catalog_role", pd.Series(index=catalog.index, dtype=object)).astype(str).eq("historical_reference")
if not historical.any():
    raise RuntimeError(
        "Scenario catalog has no historical-tail reference rows. "
        "Rerun 02_flood/03_build_event_catalog.ipynb so historical_tail_catalog.csv is regenerated and merged."
    )

required_aorc_vars = {"APCP_surface", "TMP_2maboveground", "PRES_surface", "DSWRF_surface"}
missing_sources = []
for row in catalog.to_dict("records"):
    member_file = resolve_location_path(row["rainfall_member_file"])
    event_windows_dir = member_file.parent / "event_windows"
    storm_start = pd.Timestamp(row["rainfall_member_time"]).strftime("%Y%m%dT%H")
    source_nc = event_windows_dir / f"{row['rainfall_member_id']}_{storm_start}.nc"
    if not source_nc.exists():
        missing_sources.append({"event_id": row["event_id"], "source_nc": str(source_nc), "missing": "file"})
        continue
    with xr.open_dataset(source_nc) as ds:
        missing = sorted(required_aorc_vars - set(ds.data_vars))
        empty = sorted(
            variable
            for variable in required_aorc_vars & set(ds.data_vars)
            if not bool(np.isfinite(ds[variable]).any().compute().item())
        )
    if missing or empty:
        missing_sources.append({
            "event_id": row["event_id"],
            "source_nc": str(source_nc),
            "missing": ",".join(missing),
            "no_finite_values": ",".join(empty),
        })
if missing_sources:
    raise RuntimeError(
        "Scenario catalog has AORC event-window files missing variables required by Wflow/SFINCS handoff: "
        + str(missing_sources[:10])
    )

worklist = catalog[["event_id"]].drop_duplicates()
worklist.to_csv(runtime.joint_worklist_path, index=False)
worklist.to_csv(runtime.blocked_path, index=False)

pd.Series({
    "worklist_csv": str(runtime.joint_worklist_path),
    "default_worklist_csv": str(runtime.blocked_path),
    "event_count": len(worklist),
    "historical_tail_count": int(historical.sum()),
    "aorc_event_windows_checked": len(catalog),
})


## Sync Handoff


In [ ]:
print(f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only --dry-run")
print(f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only")
print("Submit event-atomic Wflow->SFINCS jobs with cluster/run_wflow_sfincs_dsai_inland_coupled.slurm.")


## Cluster Launch Plan


In [ ]:
import json

joint_worklist_csv = runtime.joint_worklist_path
cluster_plan_path = runtime.sfincs_scenarios_root / "joint_wflow_sfincs_cluster_plan.json"
cluster_plan = {
    "workflow": "Wflow dynamic handoff -> native SFINCS staging -> SFINCS run",
    "slurm_script": "cluster/run_wflow_sfincs_dsai_inland_coupled.slurm",
    "worklist_csv": str(joint_worklist_csv),
    "FORCE_WFLOW": True,
    "OVERWRITE_METEO": True,
}
cluster_plan_path.write_text(json.dumps(cluster_plan, indent=2), encoding="utf-8")

pd.Series(cluster_plan | {"cluster_plan_json": str(cluster_plan_path)}, name="joint_wflow_sfincs_cluster_plan")


## Sample Driver QA


In [ ]:
import matplotlib.pyplot as plt

sample_event = catalog.sample(1, random_state=0).iloc[0]
member_file = resolve_location_path(sample_event["rainfall_member_file"])
event_windows_dir = member_file.parent / "event_windows"
storm_start = pd.Timestamp(sample_event["rainfall_member_time"]).strftime("%Y%m%dT%H")
rainfall_source_nc = event_windows_dir / f"{sample_event['rainfall_member_id']}_{storm_start}.nc"
if not rainfall_source_nc.exists():
    raise FileNotFoundError(rainfall_source_nc)

panels = []
with xr.open_dataset(rainfall_source_nc) as ds:
    for variable in ["APCP_surface", "TMP_2maboveground", "PRES_surface", "DSWRF_surface"]:
        if variable in ds:
            da = ds[variable]
            panels.append((variable, da.mean([dim for dim in da.dims if dim != "time"]).to_series()))

soil_moisture_path = resolve_location_path(sample_event.get("soil_moisture_member_file"))
if soil_moisture_path.exists():
    sm_time = pd.Timestamp(sample_event["soil_moisture_member_time"])
    sm = pd.read_csv(soil_moisture_path, parse_dates=["time"])
    sm_var = next((column for column in ["SOILSAT_TOP", "SOIL_M", "soil_moisture"] if column in sm), None)
    if sm_var is not None:
        sm_window = sm[sm["time"].between(sm_time - pd.Timedelta(days=7), sm_time + pd.Timedelta(days=1))]
        panels.append((f"{sm_var} around paired antecedent state", sm_window.groupby("time")[sm_var].mean()))

if not panels:
    raise RuntimeError(f"No plottable driver series for {sample_event['event_id']}")

fig, axes = plt.subplots(len(panels), 1, figsize=(10, max(2.2 * len(panels), 4)), constrained_layout=True)
if len(panels) == 1:
    axes = [axes]
for ax, (title, series) in zip(axes, panels):
    series.plot(ax=ax)
    ax.set_title(title)

pd.Series({
    "event_id": sample_event["event_id"],
    "rainfall_member_id": sample_event["rainfall_member_id"],
    "rainfall_source_nc": str(rainfall_source_nc),
    "rainfall_scale_factor": sample_event.get("rainfall_scale_factor"),
    "soil_moisture_member_id": sample_event.get("soil_moisture_member_id"),
    "wflow_event_dir": sample_event.get("wflow_event_dir"),
}, name="sample_wflow_driver_qa")
